In [8]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

In [9]:
# extract the query into df

with open(r'C:\Users\Viggo\Py\Learning Project\da_projects\Ecom-Project\queries\cohort.sql', 'r') as file: 
    sql_query = file.read()

engine = create_engine(
    "mssql+pyodbc://localhost/OlistEcom?driver=ODBC+Driver+17+for+SQL+Server"
)

df = pd.read_sql_query(sql_query, con=engine)


In [ ]:
# convert dates to date time
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['birth_date'] = pd.to_datetime(df['birth_date'])

# get the months
df['purchase_month'] = df['order_purchase_timestamp'].dt.to_period('M')
df['cohort_month'] = df['birth_date'].dt.to_period('M')

df['cohort_index'] = (df['purchase_month'] - df['cohort_month']).apply(lambda x: x.n if pd.notnull(x) else 0)

cohort_data = df.groupby(['cohort_month', 'cohort_index'])['customer_unique_id'].nunique().reset_index()

retention_matrix = cohort_data.pivot(index='cohort_month', columns='cohort_index', values='customer_unique_id')

cohort_sizes= retention_matrix.iloc[:,0]
retention_precentage = retention_matrix.divide(cohort_sizes, axis=0)


plt.figure(figsize=(16,10))
sns.heatmap(retention_precentage, annot=True, fmt='.1%', cmap='Blues', vmin=0.0, vmax=0.05)
plt.title('test')
plt.show



KeyError: 'cohort_month'